> ⚠️ **Old prototype, not maintained.** It scrapes autohero.com with Selenium, using generated class names (for example `title___1TYYE`) that change when the website is updated, so it probably no longer works. The scripts of the project do not use it.

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import pandas as pd
import time

def analyse_vin(soup):
    vins = soup.find_all('div', class_='infoContainer___1SJjz')

    data = []
    for vin in vins:
        nom_element = vin.find('h2', class_='title___1TYYE adTitle___VxmTf')
        prix_element = vin.find('div', class_='title___1TYYE price___uwAkj')
        annee_element = vin.find('li', class_='specItem___2u4I4')
        #km_element = vin.find('ul', class_='specList___1eco1')
        km_element = vin.find('div', class_='flex___Yhv67 row___2W9qg noWrap___19N-N justifySpaceBetween___uGbGu')
 
        

        # Vérifier si les éléments ont été trouvés
        if nom_element and prix_element and annee_element and km_element :
            nom = nom_element.text.strip()
            prix = prix_element.text.strip()
            km =  km_element.text.strip()
  
            data.append({'Nom': nom, 'Prix': prix,'Km': km,})

    return data


driver_path = './chromedriver'
driver = webdriver.Chrome(executable_path=driver_path)
url = 'https://www.autohero.com/fr/search/?MID=FR_SES_3000_0_11_419-0-0-0-0-0_3976-0-0-0-0-0_3&nw=g&kw=autohero&mt=e&ap=&cr=549362335189&dv=c&pl=&loc=1006094&gad_source=1&gclid=Cj0KCQjwlZixBhCoARIsAIC745DXO0mWI0DdiXjR03iFzdfkID2c_rd6YHOCjlHIZCttMK2gYzBQgXsaAkHAEALw_wcB'
driver.get(url)
time.sleep(2)

all_data = []

for _ in range(200):
    # Défilement de la page
    body = driver.find_element_by_tag_name('body')
    body.send_keys(Keys.PAGE_DOWN)
    time.sleep(1)  # Attendez un court instant entre chaque défilement

    # Récupérer le contenu de la page après le défilement
    soup = BeautifulSoup(driver.page_source, 'html.parser')

    # Appeler votre fonction de scraping avec le contenu de la page
    new_data = analyse_vin(soup)

    # Ajouter les nouvelles données à la liste de toutes les données
    all_data.extend(new_data)

# Créer un DataFrame à partir de toutes les données récupérées
df = pd.DataFrame(all_data)
print(df)

# Fermeture du navigateur
driver.quit()


In [ ]:
df

In [ ]:
# Enregistrer le DataFrame sous forme de fichier CSV
df.to_csv('data.csv', index=False)


In [ ]:
df.to_csv('../data/data.csv', index=False)
